# Imports

All needed library imports goes here.

In [1]:
from bs4 import BeautifulSoup
from datetime import datetime as dt
from pathlib import Path
from urllib.parse import urlparse
import concurrent.futures
import csv
import json
import os
import re
import requests
import time

import numpy as np
import pandas as pd

# Variables

All global variables are to be defined here.

## Process Variables

This contains all process related variables, allowing the notebook to skip parts to save time and memory.

In [2]:
topDataPresent: bool = True
"""
Determines whether the data is already present in the designated directory. This variable
dictates whether the data should be scraped or not, allowing for the program to skip the
scraping process if the data is already present.
"""

episodeDataPresent: bool = False
"""
Determines whether the episode data from `title.episode.tsv` is already present in the designated
directory. This variable dictates whether the episode data should be scraped or not, allowing for
the program to skip the scraping process if the data is already present.
"""

episodeTSV = "title.episode.tsv"
"""
Defines the TSV file where the data for episodes will be fetched.
"""

episodeCSV = "CSV/imdb_episodes.csv"
"""
Defines the CSV fir the episode data from `title.episode.tsv`.
"""

workerThreads = 10
"""
Defines the number of worker threads that will be used to scrape the data.
"""


'\nDefines the number of worker threads that will be used to scrape the data.\n'

## API Variables

All API related variables are to be defined here.

In [3]:
# Set up headers for the request
headers = {
    'Accept': '*/*',
    'Connection': 'keep-alive',
    'User-Agent': 'Mozilla/5.0 (Windows NT 6.0) AppleWebKit/537.36 (KHTML,like Gecko) Chrome/70.0.3538.110 Safari/537.36', 
    'Accept-Language': 'en-US;q=0.5,en;q=0.3',
    'Cache-Control': 'max-age=0',
    'Upgrade-Insecure-Requests': '1'
}

defaultDownloadPath = 'downloads/images'

retryDelay = 30
"""
Defines the delay in seconds before retrying the request.
"""

'\nDefines the delay in seconds before retrying the request.\n'

# Functions

All reusable functions are to be defined here.

## API Related Functions

All functions that calls or needs the API will be defined under this category.

In [4]:
def download_image(image_url: str, filename: str, folder_path: str):
	"""
	Download the image from URL and save it with show name as its filename
	
	:param image_url: URL of the image to download
	:type image_url: str
	
	:param filename: The name of that the file will use
	:type filename: str
	
	:param folder_path: Path to the folder where the image will be saved
	:type folder_path: str
	"""
	def transform(char: str):
		"""
  		A character transformer for creating a valid filename.
    
		:param char: The character to transform
		:type char: str
  		"""
		if char.isalnum() or char in (' ', '-', '_'):
			return char
		else:
			return("-")

	try:
		# Create the images directory if it doesn't exist
		Path(folder_path).mkdir(parents=True, exist_ok=True)
		
		# Clean the show name to create a valid filename
		valid_filename = "".join(transform(x) for x in filename).rstrip()
		
		# Get file extension from URL
		ext = os.path.splitext(urlparse(image_url).path)[1]
		if not ext:
			ext = '.jpg'  # Default to jpg if no extension found
			
		# Create full file path
		file_path = os.path.join(folder_path, f"{valid_filename}{ext}")
		
		# Download and save the image
		response = requests.get(image_url, headers=headers)
		response.raise_for_status()
		
		with open(file_path, 'wb') as f:
			f.write(response.content)
			
		return file_path
	except Exception as e:
		print(f"Error downloading image for {filename} ({image_url}): {str(e)} - Returning N/A")
		return "N/A"
	
def get_top_tv_shows():
	"""
	Fetches the top TV shows from the IMDB website.
	"""
	response = requests.get('https://www.imdb.com/chart/toptv/', headers=headers)
	soup = BeautifulSoup(response.content, 'html.parser')
	
	# Find the script tag containing the JSON data
	script = soup.find('script', {'type': "application/ld+json"})
	
	# Check if the script tag is found and contains data
	if script:
		data = json.loads(script.text)   
		
		# Check if 'itemListElement' exists
		if 'itemListElement' in data:
			print(json.dumps(data, indent=4))
			# Define the folder for saving images
			image_folder = 'imdb_show_images'
			# Open the CSV file in write mode
			with open('CSV/top_tv_shows_full_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
				# Create a CSV writer
				csvwriter = csv.writer(csvfile)
				
				# Write the header row (fields)
				csvwriter.writerow(['Rank', 'Show Name', 'URL', 'Description', 'Genre', 'Rating', 'Votes', 'Content Rating', 'Image URL'])
				
				# Write the data for each TV show
				for idx, item in enumerate(data['itemListElement'][:250], start=1):
					show_item = item.get('item', {})
					show_name = show_item.get('name', 'N/A')
					show_url = show_item.get('url', '')
					show_description = show_item.get('description', 'N/A')
					show_genre = show_item.get('genre', 'N/A')
					show_rating = show_item.get('aggregateRating', {}).get('ratingValue', 'N/A')
					show_votes = show_item.get('aggregateRating', {}).get('ratingCount', 'N/A')
					show_content_rating = show_item.get('contentRating', 'N/A')
					show_image_url = show_item.get('image', 'N/A')
					
					# Download the image and get local path
					local_image_path = 'N/A'
					if show_image_url != 'N/A':
						local_image_path = download_image(show_image_url, show_name, image_folder)

					# Write the row with the rank, show name, URL, description, genre, rating, votes, content rating, and image URL
					csvwriter.writerow([idx, show_name, show_url, show_description, show_genre, show_rating, show_votes, show_content_rating, show_image_url])
			
			print("Data successfully written to 'CSV/top_tv_shows_full_data.csv'.")
		else:
			print("No 'itemListElement' found in the JSON data.")
	else:
		print("Script containing JSON data not found.")

def save_episode(slug: str, csv_name: str, image_path: str, skip_non_tv_shows: bool = True):
	"""
	Saves the episode or movie information into a CSV file.
	
	:param slug: The slug of the show.
	:type slug: str
	
	:param csv_name: The name of the CSV file, including its relative path.
	:type csv_name: str
	
	:param image_path: The path where the poster will be saved.
	:type image_path: str
 
	:param skip_non_tv_shows: Whether to skip non-TV shows (e.g., movies) or not.
	:type skip_non_tv_shows: bool
	"""

	try:
		# Fetch the episode data
		response = requests.get(f'https://www.imdb.com/title/{slug}', headers=headers)
		soup = BeautifulSoup(response.content, 'html.parser')
  
		# Find the script tag containing the JSON data
		script = soup.find('script', {'type': "application/ld+json"})
  
		# Check if the script tag is found and contains data
		if script:
			data = json.loads(script.text)
   
			# Check if the data is for a TV show
			if skip_non_tv_shows:
				if data.get('@type') != 'TVSeries':
					print(f"Skipping non-TV show: {data.get('name', 'N/A')}")
					return
 
			# Return immediately if the series has no genres, rating, and date.
			genres = data.get('genre', 'N/A')
			rating = data.get('aggregateRating', {}).get('ratingValue', 'N/A')
			date = data.get('datePublished', 'N/A')
			if genres == 'N/A' or rating == 'N/A' or date == 'N/A':
				print(f"Skipping {data.get('name', 'N/A')} due to missing data.")
				return
   
			# Check if the CSV file doesn't exists, create it and write the header row
			if not os.path.exists(csv_name):
				with open(csv_name, 'w', newline = '', encoding = 'utf-8') as csvfile:
					# Creates a CSV writer
					csvwriter = csv.writer(csvfile)
	
					# Writes the header row (fields)
					csvwriter.writerow(['Slug', 'Title', 'Rating', 'Rating Count', 'Date', 'Genres', 'Poster'])
     
			# Save the poster image
			img_location = download_image(
       			data.get('image', ''),
          		slug,
            	image_path
			)
   
			# Open the CSV file in append mode
			with open(csv_name, 'a', newline = '', encoding = 'utf-8') as csvf:
				# Creates the CSV writer
				csvw = csv.writer(csvf)
    
				# Place the data into the CSV file
				# Slug, Title, Rating, Rating Count, Date, Genres, Poster
				csvw.writerow([
					slug,
					data.get('name', 'N/A'),
					rating,
					date,
					genres,
					img_location
				])
		else:
			print("Script containing JSON data not found.")
	except Exception as e:
		print(f"Error saving episode: {str(e)}")



## Process Functions

Functions that handles repeatable processes.

In [5]:
def read_tsv_file(file: str):
    """
    Read and returns the data from the TSV file as dataframes.
    
    :param file: The TSV file to read. Must be a relative or absolute path.
    :type file: str
    
    :return: The data from the TSV file as a pandas dataframe.
    """
    
    if not os.path.exists(file):
        raise FileNotFoundError(f"File '{file}' not found.")
    
    # Read the TSV file
    data = pd.read_csv(file, sep = '\t', low_memory = False)
    
    return data

def get_episode_data(chunk: np.ndarray, csv_name: str, image_path: str, savedEpisodes: pd.DataFrame, skipExisting: bool = True):
	"""
	Scrapes the episode data for the given chunk of data.
	
	:param chunk: The chunk of data to scrape.
	:type chunk: np.ndarray
	
	:param csv_name: The name of the CSV file, including its relative path.
	:type csv_name: str
	
	:param image_path: The path where the poster will be saved.
	:type image_path: str
 
	:param savedEpisodes: The episodes that have already been saved.
	:type savedEpisodes: pd.DataFrame
 
	:param skipExisting: Whether to skip existing data or not.
	:type skipExisting: bool
	"""
	
	for slug in chunk:
		if skipExisting:
			if savedEpisodes is not None and slug in savedEpisodes['Slug'].values:
				print(f"Skipping existing episode with slug '{slug}'")
				continue
		
		save_episode(slug, csv_name, image_path)
  
def is_tv_show(slug: str):
	"""
 	Checks if the slug provided for the IMDb show is a TV show.

	:param slug: The slug of the show.
	:type slug: str

	:return: Whether the slug is a TV show or not.
	"""
 
	try:
		response = requests.get(f"https://www.imdb.com/title/{slug}", headers=headers)
		soup = BeautifulSoup(response.content, 'html.parser')
  
		# Find the metadata fir the type of the show
		meta = soup.find("script",  type="application/ld+json")
		if not meta:
			return False

		data = json.loads(meta.text)
		if data.get("@type") == "TVSeries":
			return True
	except Exception as e:
		print(f"Error checking if '{slug}' is a TV show: {e}")
		return False
	return False



# Data Scraping/Gathering

This is where data is gathered. If the data is already present, certain variables can be set to `True` to skip the scraping process for that particular part.

## Rate Limit Check

Checks whether the IMDB is currently rate limited. This code block will continue to check the IMDB website every `retryDelay` seconds until its either stopped or
has returned a status code of `200`.

In [6]:
response = requests.get('https://datasets.imdbws.com/', headers=headers)

while response.status_code != 200:
	print(f"Failed to fetch the data from the URL. Retrying in {retryDelay} seconds...")
	time.sleep(retryDelay)
	response = requests.get('https://datasets.imdbws.com/', headers=headers)
 
print(f"Successfully fetched the data from the URL. Status code: {response.status_code}")

Successfully fetched the data from the URL. Status code: 200


## Top TV Shows

This part fetches the top TV shows. This part is handled by the `topDataPresent` variable; defined at the top of the notebook to determine whether this data should be scraped or not.

In [7]:
if not topDataPresent:
	get_top_tv_shows()
else:
    print("Top TV show data already present. Skipping scraping process.")

Top TV show data already present. Skipping scraping process.


## Episode TSV

This part fetches the episodes from a TSV file `title.episode.tsv`. This part is handled by the `episodeDataPresent` variable; defined at the top of the notebook to determine whether this data should be scraped or not.

In [ ]:
if not episodeDataPresent:
	imdbItems = read_tsv_file(f"IMDB_TSV/{episodeTSV}")
	print(imdbItems)

	episodeSlugs = imdbItems['parentTconst']
	np.append(episodeSlugs, imdbItems['tconst'])
	print(episodeSlugs)
 
	# Remove duplicates
	episodeSlugs = np.unique(episodeSlugs)
 
	# Check if the CSV file already exists, save the data as DataFrame
	savedEps = None
	if os.path.exists(episodeCSV):
		savedEps = pd.read_csv(episodeCSV)
		print(savedEps)
  
	# Split the episode slugs into n chunks for parallel processing (Defined by `workerThreads` variable)
	chunks = np.array_split(episodeSlugs, workerThreads)
	print(chunks)
 
	# Create a thread for each chunk
	pool = concurrent.futures.ThreadPoolExecutor(max_workers = workerThreads)
 
	for chunk in chunks:
		pool.submit(get_episode_data, chunk, episodeCSV, 'imdb_episode_images', savedEps)
	pool.shutdown(wait = True)
  
else:
    print("Episode data already present. Skipping scraping process.")

            tconst parentTconst seasonNumber episodeNumber
0        tt0031458   tt32857063           \N            \N
1        tt0041951    tt0041038            1             9
2        tt0042816    tt0989125            1            17
3        tt0042889    tt0989125           \N            \N
4        tt0043426    tt0040051            3            42
...            ...          ...          ...           ...
8792019  tt9916846    tt1289683            3            18
8792020  tt9916848    tt1289683            3            17
8792021  tt9916850    tt1289683            3            19
8792022  tt9916852    tt1289683            3            20
8792023  tt9916880    tt0985991            4             2

[8792024 rows x 4 columns]
0          tt32857063
1           tt0041038
2           tt0989125
3           tt0989125
4           tt0040051
              ...    
8792019     tt1289683
8792020     tt1289683
8792021     tt1289683
8792022     tt1289683
8792023     tt0985991
Name: parentTconst, Le